<a href="https://colab.research.google.com/github/clistenesrodger/Trabalho-Final-Sistemas-Inteligentes/blob/main/Trabalho_final_de_SI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classificação de Imagens por CNN

Este notebook foi organizado em seções separadas por arquitetura de CNN para facilitar a execução no Google Colab.

Modelos incluídos:

- DenseNet121: baseline do artigo original
- ResNet50: CNN clássica e muito citada
- ConvNeXt V2: CNN moderna
- InceptionV3: arquitetura com conceito diferente

A ideia é manter a base comum no início e deixar cada modelo em um bloco próprio.

In [1]:
# ============================================================
# Base comum para todos os experimentos Versão final
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd
import kagglehub
from PIL import Image
from torchvision import transforms as T

from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# ============================================================
# 2. DOWNLOAD E LEITURA DO DATASET
# ============================================================

path = kagglehub.dataset_download("mahyeks/almond-varieties")
dataset_dir = os.path.join(path, "dataset")
valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

classes = sorted(
    nome for nome in os.listdir(dataset_dir)
    if os.path.isdir(os.path.join(dataset_dir, nome))
)

image_paths, labels = [], []
for classe in classes:
    classe_dir = os.path.join(dataset_dir, classe)
    for nome_arquivo in sorted(os.listdir(classe_dir)):
        if nome_arquivo.lower().endswith(valid_extensions):
            image_paths.append(os.path.join(classe_dir, nome_arquivo))
            labels.append(classe)

base_df = pd.DataFrame({"image_path": image_paths, "label": labels})
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(base_df["label"])

print("Base carregada com sucesso.")
print(base_df["label"].value_counts())

# ============================================================
# 3. DIVISÃO TREINO / TESTE (somente imagens originais)
# ============================================================

X_train_orig_paths, X_test_paths, y_train_orig, y_test = train_test_split(
    base_df["image_path"].values,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"\nDivisão treino/teste (originais):")
print(f"  Treino : {len(X_train_orig_paths)} imagens")
print(f"  Teste  : {len(X_test_paths)} imagens")

# ============================================================
# 4. SPLIT TREINO / VALIDAÇÃO ANTES DA AUGMENTATION
#    StratifiedGroupKFold: mesma imagem base nunca aparece
#    em treino e validação ao mesmo tempo (evita vazamento).
# ============================================================

SUFIXOS_AUG = ["_hflip", "_vflip", "_rot30", "_jitter", "_blur", "_affine"]


def extrair_nome_base(path):
    """Nome base da imagem, sem extensão e sem sufixo de augmentation."""
    nome_arquivo = os.path.splitext(os.path.basename(path))[0]
    for suf in SUFIXOS_AUG:
        if nome_arquivo.endswith(suf):
            return nome_arquivo[: -len(suf)]
    return nome_arquivo


groups_orig = np.array([extrair_nome_base(p) for p in X_train_orig_paths])
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, val_idx = next(sgkf.split(X_train_orig_paths, y_train_orig, groups_orig))

X_tr_orig = X_train_orig_paths[tr_idx]
y_tr_orig = y_train_orig[tr_idx]
X_val_paths = X_train_orig_paths[val_idx]
y_val = y_train_orig[val_idx]

print(f"\nSplit treino/validação (só originais, antes do aug):")
print(f"  Treino (orig)    : {len(X_tr_orig)} imagens")
print(f"  Validação (orig) : {len(X_val_paths)} imagens")

# ============================================================
# 5. DATA AUGMENTATION (somente no fold de treino)
#    Validação e teste ficam sem augmentation.
#    Flips via transforms funcionais; demais ops com seed por imagem.
# ============================================================

import torch


def _aplicar_transform(img, transform, rng):
    """Aplica transform estocástico com seed reprodutível por imagem."""
    torch.manual_seed(int(rng.integers(0, 2**31)))
    np.random.seed(int(rng.integers(0, 2**31)))
    return transform(img)


stochastic_ops = {
    "rot30":  T.RandomRotation(degrees=(-30, 30)),
    "jitter": T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    "blur":   T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    "affine": T.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
}

aug_dir = "augmented_train"
os.makedirs(aug_dir, exist_ok=True)

augmented_paths  = []
augmented_labels = []
total_originais  = len(X_tr_orig)

print(f"\nAplicando data augmentation em {total_originais} imagens de treino...")
print(f"Operações: hflip, vflip + {list(stochastic_ops.keys())}\n")

for idx, (img_path, label) in enumerate(zip(X_tr_orig, y_tr_orig)):
    img       = Image.open(img_path).convert("RGB")
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    rng       = np.random.default_rng(abs(hash(base_name)) % (2**31))

    for op_name, aug_img in [
        ("hflip", T.functional.hflip(img)),
        ("vflip", T.functional.vflip(img)),
    ]:
        save_path = os.path.join(aug_dir, f"{base_name}_{op_name}.jpg")
        aug_img.save(save_path, quality=95)
        augmented_paths.append(save_path)
        augmented_labels.append(label)

    for op_name, transform in stochastic_ops.items():
        aug_img   = _aplicar_transform(img, transform, rng)
        save_path = os.path.join(aug_dir, f"{base_name}_{op_name}.jpg")
        aug_img.save(save_path, quality=95)
        augmented_paths.append(save_path)
        augmented_labels.append(label)

    if (idx + 1) % 100 == 0 or (idx + 1) == total_originais:
        print(f"  ✓ {idx + 1}/{total_originais} imagens processadas")

# Treino final = originais do fold de treino + variantes augmentadas
X_tr_paths = np.concatenate([X_tr_orig, np.array(augmented_paths)])
y_tr       = np.concatenate([y_tr_orig, np.array(augmented_labels)])

# Aliases usados pelos blocos de modelos
X_train_paths = X_tr_paths
y_train       = y_tr

# ============================================================
# 6. RESUMO FINAL
# ============================================================

print(f"\n{'=' * 50}")
print("RESUMO DO DATASET")
print(f"{'=' * 50}")
print(f"Imagens originais de treino : {total_originais}")
print(f"Imagens geradas (aug)       : {len(augmented_paths)}")
print(f"Total de treino (com aug)   : {len(X_tr_paths)}")
print(f"Validação (sem aug)         : {len(X_val_paths)}")
print(f"Total de teste              : {len(X_test_paths)}")

print(f"\nDistribuição por classe (treino + aug):")
unique, counts = np.unique(y_tr, return_counts=True)
for cls_idx, count in zip(unique, counts):
    print(f"  {label_encoder.classes_[cls_idx]:<20}: {count}")

Using Colab cache for faster access to the 'almond-varieties' dataset.
Base carregada com sucesso.
label
KAPADOKYA    465
AK           401
SIRA         384
NURLU        306
Name: count, dtype: int64

Divisão treino/teste (originais):
  Treino : 1244 imagens
  Teste  : 312 imagens

Split treino/validação (só originais, antes do aug):
  Treino (orig)    : 994 imagens
  Validação (orig) : 250 imagens

Aplicando data augmentation em 994 imagens de treino...
Operações: hflip, vflip + ['rot30', 'jitter', 'blur', 'affine']

  ✓ 100/994 imagens processadas
  ✓ 200/994 imagens processadas
  ✓ 300/994 imagens processadas
  ✓ 400/994 imagens processadas
  ✓ 500/994 imagens processadas
  ✓ 600/994 imagens processadas
  ✓ 700/994 imagens processadas
  ✓ 800/994 imagens processadas
  ✓ 900/994 imagens processadas
  ✓ 994/994 imagens processadas

RESUMO DO DATASET
Imagens originais de treino : 994
Imagens geradas (aug)       : 5964
Total de treino (com aug)   : 6958
Validação (sem aug)         : 250


## DenseNet121

Baseline do artigo original. Use este bloco para a primeira comparação.

In [ ]:
# ============================================================
# DenseNet121 — Fine-tuning com carregamento lazy (tf.data)
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import json
import pickle
import numpy as np
from datetime import datetime

import tensorflow as tf
from tensorflow.keras.applications.densenet import DenseNet121, preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import backend as K

from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, roc_auc_score, classification_report,
)

# ============================================================
# 2. LIBERAÇÃO DE MEMÓRIA
# ============================================================

try:
    del model
    K.clear_session()
    gc.collect()
    print("✓ Memória do modelo anterior liberada")
except Exception:
    pass

# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

def load_and_preprocess_image(path, label):
    """
    Carrega e pré-processa UMA imagem sob demanda.
    Chamada pelo tf.data somente quando o batch for necessário —
    nunca carrega o dataset inteiro na RAM.
    """
    img_raw = tf.io.read_file(path)
    img = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])                     # necessário após decode_image
    img = tf.image.resize(img, [224, 224])
    img = preprocess_input(tf.cast(img, tf.float32))   # normalização DenseNet
    return img, label


def criar_dataset(paths, labels, batch_size=32, shuffle=False):
    """
    Cria um tf.data.Dataset com carregamento lazy em disco.

    RAM utilizada por vez:
        batch_size × 224 × 224 × 3 × 4 bytes
        Ex.: 32 imagens ≈ 19 MB  (vs. dataset inteiro na RAM antes)
    """
    dataset = tf.data.Dataset.from_tensor_slices(
        (list(map(str, paths)), labels.astype(np.int32))
    )
    if shuffle:
        dataset = dataset.shuffle(buffer_size=2000, seed=42, reshuffle_each_iteration=True)
    return (
        dataset
        .map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)   # pré-carrega o próximo batch enquanto a GPU treina
    )


def calcular_metricas(y_true, y_pred, y_pred_proba=None, num_classes=None):
    """Calcula métricas de classificação e retorna dicionário."""
    metricas = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall":    recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1-Score":  f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "Kappa":     cohen_kappa_score(y_true, y_pred),
    }
    if y_pred_proba is not None:
        if num_classes is None:
            num_classes = y_pred_proba.shape[1]
        if num_classes == 2:
            metricas["AUC"] = roc_auc_score(y_true, y_pred_proba[:, 1])
        else:
            y_true_bin = label_binarize(y_true, classes=range(num_classes))
            metricas["AUC"] = roc_auc_score(
                y_true_bin, y_pred_proba, multi_class="ovr", average="weighted"
            )
    else:
        metricas["AUC"] = None
    return metricas


def salvar_resultados(nome_modelo, metricas, y_true, y_pred, y_pred_proba,
                      history=None, diretorio="resultados"):
    """Salva métricas, predições e relatório em arquivos locais."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pasta = os.path.join(diretorio, f"{nome_modelo}_{timestamp}")
    os.makedirs(pasta, exist_ok=True)

    metricas_serial = {
        k: (float(v) if hasattr(v, "item") else v)
        for k, v in metricas.items()
    }

    with open(os.path.join(pasta, "metricas.json"), "w") as f:
        json.dump({"modelo": nome_modelo, "data_execucao": timestamp,
                   "metricas": metricas_serial}, f, indent=4)

    np.save(os.path.join(pasta, "y_true.npy"), y_true)
    np.save(os.path.join(pasta, "y_pred.npy"), y_pred)
    np.save(os.path.join(pasta, "y_pred_proba.npy"), y_pred_proba)

    with open(os.path.join(pasta, "classification_report.json"), "w") as f:
        json.dump(classification_report(y_true, y_pred, output_dict=True), f, indent=4)

    if history is not None:
        hist_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
        with open(os.path.join(pasta, "training_history.json"), "w") as f:
            json.dump(hist_dict, f, indent=4)
        with open(os.path.join(pasta, "training_history.pkl"), "wb") as f:
            pickle.dump(history.history, f)

    with open(os.path.join(pasta, "resumo.txt"), "w", encoding="utf-8") as f:
        f.write("=" * 50 + "\n")
        f.write(f"RESUMO DO MODELO: {nome_modelo}\n")
        f.write("=" * 50 + "\n\n")
        f.write("MÉTRICAS:\n" + "-" * 30 + "\n")
        for metrica, valor in metricas_serial.items():
            f.write(
                f"{metrica}: {valor:.4f}\n" if valor is not None
                else f"{metrica}: Não disponível\n"
            )

    print(f"\n✓ Resultados salvos em: {pasta}")
    for arq in ["metricas.json", "y_true/pred.npy", "y_pred_proba.npy",
                "classification_report.json", "training_history.json", "resumo.txt"]:
        print(f"  - {arq}")

    return pasta


def criar_loss(num_classes, label_smoothing=0.1):
    """Loss com label smoothing; fallback para TF antigo sem esse parâmetro."""
    try:
        return tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=label_smoothing)
    except TypeError:
        smooth = label_smoothing

        def loss_fn(y_true, y_pred):
            y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
            y_true_oh = tf.one_hot(y_true, num_classes)
            y_true_oh = y_true_oh * (1.0 - smooth) + smooth / float(num_classes)
            return tf.reduce_mean(tf.keras.losses.categorical_crossentropy(y_true_oh, y_pred))

        return loss_fn


# ============================================================
# 4. CONJUNTOS TREINO / VALIDAÇÃO / TESTE
#    Split por grupos e augmentation já feitos na célula base.
#    Validação contém apenas imagens originais (sem vazamento).
# ============================================================

print(f"Treino    : {len(X_tr_paths):>6} imagens")
print(f"Validação : {len(X_val_paths):>6} imagens (somente originais)")
print(f"Teste     : {len(X_test_paths):>6} imagens")

# ============================================================
# 5. DATASETS LAZY
# ============================================================

BATCH_SIZE = 32

train_ds = criar_dataset(X_tr_paths, y_tr,       batch_size=BATCH_SIZE, shuffle=True)
val_ds   = criar_dataset(X_val_paths, y_val,     batch_size=BATCH_SIZE)
test_ds  = criar_dataset(X_test_paths, y_test,   batch_size=BATCH_SIZE)

ram_por_batch = BATCH_SIZE * 224 * 224 * 3 * 4 / 1e6
print(f"\n✓ Datasets criados com carregamento lazy")
print(f"  RAM por batch : ~{ram_por_batch:.0f} MB  (antes: dataset inteiro)")

# ============================================================
# 6. CONSTRUÇÃO DO MODELO
# ============================================================

print("\n" + "=" * 60)
print("CONSTRUÇÃO DO MODELO — DenseNet121")
print("=" * 60)

base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

x      = GlobalAveragePooling2D()(base_model.output)
x      = Dropout(0.3)(x)
output = Dense(len(label_encoder.classes_), activation="softmax")(x)
model  = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=criar_loss(len(label_encoder.classes_)),
    metrics=["accuracy"],
)

treinavel = sum(p.numpy().size for p in model.trainable_weights)
total     = sum(p.numpy().size for p in model.weights)
print(f"✓ Parâmetros treináveis : {treinavel:,}  /  {total:,} total")

# ============================================================
# 7. TREINAMENTO
# ============================================================

os.makedirs("models", exist_ok=True)

callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=7,
        restore_best_weights=True, verbose=1,
    ),
    ModelCheckpoint(
        "models/densenet121_best.keras", monitor="val_accuracy",
        save_best_only=True, verbose=0,
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3,
        min_lr=1e-7, verbose=1,
    ),
]

print("\n" + "=" * 60)
print("TREINAMENTO — DenseNet121")
print("=" * 60)

history = model.fit(
    train_ds,
    validation_data=val_ds,    # validation_split substituído por validation_data
    epochs=50,
    callbacks=callbacks,
    verbose=1,
)

model.save("models/densenet121.keras")
print("✓ Modelo salvo em: models/densenet121.keras")

# ============================================================
# 8. AVALIAÇÃO
# ============================================================

print("\n" + "=" * 60)
print("AVALIAÇÃO DO MODELO")
print("=" * 60)

# predict() sobre dataset: processa batch a batch, sem carregar tudo na RAM
y_pred_proba = model.predict(test_ds, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)

metricas = calcular_metricas(
    y_test, y_pred, y_pred_proba,
    num_classes=len(label_encoder.classes_),
)

print("\n" + "=" * 50)
print("MÉTRICAS — DenseNet121")
print("=" * 50)
for metrica, valor in metricas.items():
    if valor is not None:
        print(f"  {metrica:<12}: {valor:.4f}")
    else:
        print(f"  {metrica:<12}: Não disponível")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# ============================================================
# 9. SALVAR RESULTADOS
# ============================================================

pasta_resultados = salvar_resultados(
    nome_modelo="DenseNet121",
    metricas=metricas,
    y_true=y_test,
    y_pred=y_pred,
    y_pred_proba=y_pred_proba,
    history=history,
    diretorio="resultados",
)

# ============================================================
# 10. RESUMO FINAL
# ============================================================

print(f"\n{'=' * 60}")
print("✅ RESUMO FINAL — DenseNet121")
print(f"{'=' * 60}")
print(f"✓ Acurácia : {metricas['Accuracy']:.4f}")
print(f"✓ F1-Score : {metricas['F1-Score']:.4f}")
print(f"✓ AUC      : {metricas['AUC']:.4f}" if metricas["AUC"] else "✓ AUC: Não disponível")
print(f"✓ Kappa    : {metricas['Kappa']:.4f}")
print(f"✓ Pasta    : {pasta_resultados}")

K.clear_session()
gc.collect()
print("\n✓ Sessão Keras limpa")

Treino    :   6958 imagens
Validação :    250 imagens (somente originais)
Teste     :    312 imagens

✓ Datasets criados com carregamento lazy
  RAM por batch : ~19 MB  (antes: dataset inteiro)

CONSTRUÇÃO DO MODELO — DenseNet121
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✓ Parâmetros treináveis : 4,100  /  7,041,604 total

TREINAMENTO — DenseNet121
Epoch 1/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 125s 389ms/step - accuracy: 0.3580 - loss: 1.5331 - val_accuracy: 0.7320 - val_loss: 1.0045 - learning_rate: 1.0000e-04
Epoch 2/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 25s 113ms/step - accuracy: 0.5867 - loss: 1.0706 - val_accuracy: 0.8960 - val_loss: 0.7475 - learning_rate: 1.0000e-04
Epoch 3/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 41s 112ms/step - accuracy: 0.7436 - loss: 0.8308 - val_accuracy: 0.9240 - val_loss: 0.6225 - learning_rate: 1.0000e-04
Epoch 4/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 24s 110ms/step - accuracy: 0.8227 - loss: 0.7184 - val_accuracy: 0.9360 - val_loss: 0.5604 - learning_rate: 1.0000e-04
Epo

In [ ]:
# ============================================
# VISUALIZAÇÃO DOS RESULTADOS - DenseNet121
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import json
import os
from pathlib import Path

# Configurar estilo dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# ============================================
# 1. CARREGAR RESULTADOS SALVOS
# ============================================
def carregar_resultados(pasta_resultados):
    """Carrega os resultados salvos do modelo"""
    resultados = {}

    # Carregar métricas
    with open(os.path.join(pasta_resultados, "metricas.json"), 'r') as f:
        resultados['metricas'] = json.load(f)

    # Carregar arrays numpy
    resultados['y_true'] = np.load(os.path.join(pasta_resultados, "y_true.npy"))
    resultados['y_pred'] = np.load(os.path.join(pasta_resultados, "y_pred.npy"))
    resultados['y_pred_proba'] = np.load(os.path.join(pasta_resultados, "y_pred_proba.npy"))

    # Carregar classification report
    with open(os.path.join(pasta_resultados, "classification_report.json"), 'r') as f:
        resultados['classification_report'] = json.load(f)

    # Carregar histórico se existir
    history_path = os.path.join(pasta_resultados, "training_history.json")
    if os.path.exists(history_path):
        with open(history_path, 'r') as f:
            resultados['history'] = json.load(f)

    return resultados

# Carregar dados
print("Carregando resultados salvos...")
resultados = carregar_resultados(pasta_resultados)
print("✓ Resultados carregados com sucesso!")

# ============================================
# 2. TABELA DE MÉTRICAS
# ============================================
print("\n" + "="*60)
print("📊 TABELA DE MÉTRICAS - DenseNet121")
print("="*60)

# Criar DataFrame com as métricas
metricas_dict = resultados['metricas']['metricas']
df_metricas = pd.DataFrame({
    'Métrica': list(metricas_dict.keys()),
    'Valor': list(metricas_dict.values())
})

# Formatar valores
def format_bar(value):
    """Cria uma barra de progresso visual"""
    if value is None or value == "N/A":
        return "Não disponível"
    try:
        val = float(value)
        bar_length = int(val * 20)  # 20 caracteres máximos
        bar = "█" * bar_length + "░" * (20 - bar_length)
        return f"{bar} {val:.4f}"
    except:
        return str(value)

df_metricas['Visualização'] = df_metricas['Valor'].apply(format_bar)

# Exibir tabela formatada
print("\nMÉTRICAS DE DESEMPENHO:")
print("-"*60)
for idx, row in df_metricas.iterrows():
    print(f"  {row['Métrica']:<12} {row['Visualização']}")
print("-"*60)

# ============================================
# 3. MATRIZ DE CONFUSÃO
# ============================================
print("\n" + "="*60)
print("🎯 MATRIZ DE CONFUSÃO - DenseNet121")
print("="*60)

# Obter classes
if hasattr(label_encoder, 'classes_'):
    nomes_classes = [str(c) for c in label_encoder.classes_]
else:
    # Fallback: deduzir classes únicas
    nomes_classes = sorted(set(str(c) for c in np.unique(np.concatenate([resultados['y_true'], resultados['y_pred']]))))

n_classes = len(nomes_classes)
print(f"Classes detectadas: {n_classes}")
print(f"Nomes: {', '.join(nomes_classes[:5])}{'...' if len(nomes_classes) > 5 else ''}")

# Calcular matriz de confusão
cm = confusion_matrix(resultados['y_true'], resultados['y_pred'])

# Criar figura com subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Para muitas classes, ajustar tamanho da fonte
if n_classes > 10:
    annot_fontsize = 8
    rotation_angle = 45
else:
    annot_fontsize = 10
    rotation_angle = 0

# ===== Matriz de Confusão - Contagens =====
sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=nomes_classes,
            yticklabels=nomes_classes,
            ax=axes[0],
            annot_kws={'size': annot_fontsize},
            cbar_kws={'label': 'Contagem'})
axes[0].set_title('Matriz de Confusão - Contagens', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Classe Predita', fontsize=12)
axes[0].set_ylabel('Classe Verdadeira', fontsize=12)
axes[0].tick_params(axis='x', rotation=rotation_angle)
axes[0].tick_params(axis='y', rotation=0)

# ===== Matriz de Confusão - Percentuais =====
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_percent,
            annot=True,
            fmt='.1f',
            cmap='RdYlGn',
            xticklabels=nomes_classes,
            yticklabels=nomes_classes,
            ax=axes[1],
            vmin=0,
            vmax=100,
            annot_kws={'size': annot_fontsize},
            cbar_kws={'label': '%'})
axes[1].set_title('Matriz de Confusão - Percentuais (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Classe Predita', fontsize=12)
axes[1].set_ylabel('Classe Verdadeira', fontsize=12)
axes[1].tick_params(axis='x', rotation=rotation_angle)
axes[1].tick_params(axis='y', rotation=0)

plt.suptitle('DenseNet121 - Análise de Classificação', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(pasta_resultados, 'matriz_confusao.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Matriz de confusão salva em: matriz_confusao.png")

# ============================================
# 4. MÉTRICAS POR CLASSE (CORRIGIDO)
# ============================================
print("\n" + "="*60)
print("📈 MÉTRICAS POR CLASSE")
print("="*60)

# Extrair métricas por classe do classification report
report = resultados['classification_report']

# Filtrar apenas as entradas numéricas (classes)
classes_no_report = [k for k in report.keys() if k not in ['accuracy', 'macro avg', 'weighted avg'] and isinstance(report[k], dict)]

if not classes_no_report:
    print("⚠️ Nenhuma classe individual encontrada no report. Usando índices numéricos...")
    # Criar mapeamento se necessário
    classes_no_report = [str(c) for c in sorted(set(resultados['y_true']))]

    # Recalcular métricas por classe se não estiverem no report
    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, support = precision_recall_fscore_support(
        resultados['y_true'],
        resultados['y_pred'],
        average=None
    )

    report_por_classe = {}
    for i, classe in enumerate(classes_no_report):
        report_por_classe[classe] = {
            'precision': precision[i] if i < len(precision) else 0,
            'recall': recall[i] if i < len(recall) else 0,
            'f1-score': f1[i] if i < len(f1) else 0,
            'support': support[i] if i < len(support) else 0
        }
else:
    report_por_classe = {classe: report[classe] for classe in classes_no_report}

# Criar DataFrame com métricas por classe
df_por_classe = pd.DataFrame()
for classe, metrics in report_por_classe.items():
    df_por_classe[classe] = {
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1-Score': metrics['f1-score'],
        'Support': metrics['support']
    }

df_por_classe = df_por_classe.round(3)

# Adicionar médias
for avg_type in ['macro avg', 'weighted avg']:
    if avg_type in report:
        df_por_classe[avg_type] = {
            'Precision': report[avg_type]['precision'],
            'Recall': report[avg_type]['recall'],
            'F1-Score': report[avg_type]['f1-score'],
            'Support': report[avg_type]['support']
        }

print("\nTabela de métricas por classe:")
print("="*60)
# Mostrar apenas as primeiras e últimas classes se houver muitas
if len(df_por_classe.columns) > 10:
    print("(Mostrando primeiras 5 e últimas 5 classes)")
    pd.set_option('display.max_columns', 12)
print(df_por_classe.to_string())
print("="*60)

# ============================================
# 5. RESUMO VISUAL (CORRIGIDO)
# ============================================
print("\n" + "="*60)
print("🎯 RESUMO DO DESEMPENHO")
print("="*60)

# Encontrar melhor e pior classe usando o report_por_classe
if report_por_classe:
    # Ordenar classes por F1-Score
    f1_scores = {classe: metrics['f1-score'] for classe, metrics in report_por_classe.items()}
    if f1_scores:
        melhor_classe = max(f1_scores, key=f1_scores.get)
        pior_classe = min(f1_scores, key=f1_scores.get)

        print(f"✓ Acurácia geral: {metricas_dict['Accuracy']:.4f}")
        print(f"✓ Número de classes: {len(report_por_classe)}")
        print(f"✓ Melhor classe: {melhor_classe} (F1-Score: {f1_scores[melhor_classe]:.3f})")
        print(f"✓ Pior classe: {pior_classe} (F1-Score: {f1_scores[pior_classe]:.3f})")
        print(f"✓ Total de imagens de teste: {len(resultados['y_true'])}")

        # Distribuição das classes
        print(f"\nDistribuição - Top 5 classes com melhor F1-Score:")
        top5 = sorted(f1_scores.items(), key=lambda x: x[1], reverse=True)[:5]
        for i, (classe, f1) in enumerate(top5, 1):
            print(f"  {i}. {classe}: {f1:.3f}")

        print(f"\nDistribuição - Top 5 classes com pior F1-Score:")
        bottom5 = sorted(f1_scores.items(), key=lambda x: x[1])[:5]
        for i, (classe, f1) in enumerate(bottom5, 1):
            print(f"  {i}. {classe}: {f1:.3f}")
else:
    print("⚠️ Não foi possível extrair métricas por classe.")

# Salvar tabelas como CSV
df_metricas.to_csv(os.path.join(pasta_resultados, 'tabela_metricas.csv'), index=False)
df_por_classe.to_csv(os.path.join(pasta_resultados, 'metricas_por_classe.csv'))

print(f"\n✓ Tabelas salvas em: {pasta_resultados}")
print(f"  - tabela_metricas.csv")
print(f"  - metricas_por_classe.csv")
print(f"  - matriz_confusao.png")

# ============================================
# 6. GRÁFICO DE BARRAS DAS MÉTRICAS
# ============================================
print("\n" + "="*60)
print("📊 GERANDO GRÁFICO DE BARRAS")
print("="*60)

# Filtrar apenas métricas numéricas
metricas_plot = {k: v for k, v in metricas_dict.items() if v is not None}

if metricas_plot:
    fig, ax = plt.subplots(figsize=(10, 6))

    # Criar barras com gradiente de cores
    n_barras = len(metricas_plot)
    cores = plt.cm.viridis(np.linspace(0.2, 0.9, n_barras))
    barras = ax.bar(range(n_barras),
                    list(metricas_plot.values()),
                    color=cores,
                    edgecolor='black',
                    linewidth=1.5)

    # Adicionar valores nas barras
    for barra, (nome, valor) in zip(barras, metricas_plot.items()):
        height = barra.get_height()
        ax.text(barra.get_x() + barra.get_width()/2., height + 0.01,
                f'{valor:.4f}',
                ha='center', va='bottom', fontweight='bold', fontsize=11)

    # Configurar eixos
    ax.set_xticks(range(n_barras))
    ax.set_xticklabels(list(metricas_plot.keys()), rotation=45, ha='right')
    ax.set_ylabel('Valor', fontsize=12, fontweight='bold')
    ax.set_title('DenseNet121 - Métricas de Desempenho', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)

    # Adicionar linhas de referência
    ax.axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Excelente (0.8)')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Bom (0.6)')
    ax.axhline(y=0.4, color='red', linestyle='--', alpha=0.5, label='Regular (0.4)')
    ax.legend(loc='lower left')

    plt.tight_layout()
    plt.savefig(os.path.join(pasta_resultados, 'grafico_metricas.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Gráfico de métricas salvo em: grafico_metricas.png")
else:
    print("⚠️ Nenhuma métrica disponível para plotar.")

print("\n✅ Visualização completa dos resultados!")

## ResNet50

CNN clássica e muito citada. Use este bloco para o segundo experimento.

In [ ]:
# ============================================================
# ResNet50 — Fine-tuning com carregamento lazy (tf.data)
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import json
import pickle
import numpy as np
from datetime import datetime

import tensorflow as tf
# Alias para não colidir com o preprocess_input do DenseNet121
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input as resnet_preprocess
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import backend as K

from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, roc_auc_score, classification_report,
)

# ============================================================
# 2. LIBERAÇÃO DE MEMÓRIA
# ============================================================

try:
    del model
    K.clear_session()
    gc.collect()
    print("✓ Memória do modelo anterior liberada")
except Exception:
    pass

# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

def load_and_preprocess_resnet(path, label):
    """
    Carrega e pré-processa UMA imagem com normalização ResNet50.
    Chamada pelo tf.data somente quando o batch for necessário —
    nunca carrega o dataset inteiro na RAM.

    ResNet50 usa normalização diferente da DenseNet121:
    subtrai a média de cada canal BGR (ImageNet), sem escala.
    """
    img_raw = tf.io.read_file(path)
    img = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [224, 224])
    img = resnet_preprocess(tf.cast(img, tf.float32))
    return img, label


def criar_dataset_resnet(paths, labels, batch_size=32, shuffle=False):
    """
    Cria um tf.data.Dataset com carregamento lazy em disco.

    RAM utilizada por vez:
        batch_size × 224 × 224 × 3 × 4 bytes
        Ex.: 32 imagens ≈ 19 MB  (vs. dataset inteiro na RAM antes)
    """
    dataset = tf.data.Dataset.from_tensor_slices(
        (list(map(str, paths)), labels.astype(np.int32))
    )
    if shuffle:
        dataset = dataset.shuffle(buffer_size=2000, seed=42, reshuffle_each_iteration=True)
    return (
        dataset
        .map(load_and_preprocess_resnet, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )


def calcular_metricas(y_true, y_pred, y_pred_proba=None, num_classes=None):
    """Calcula métricas de classificação e retorna dicionário."""
    metricas = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall":    recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1-Score":  f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "Kappa":     cohen_kappa_score(y_true, y_pred),
    }
    if y_pred_proba is not None:
        if num_classes is None:
            num_classes = y_pred_proba.shape[1]
        if num_classes == 2:
            metricas["AUC"] = roc_auc_score(y_true, y_pred_proba[:, 1])
        else:
            y_true_bin = label_binarize(y_true, classes=range(num_classes))
            metricas["AUC"] = roc_auc_score(
                y_true_bin, y_pred_proba, multi_class="ovr", average="weighted"
            )
    else:
        metricas["AUC"] = None
    return metricas


def salvar_resultados_resnet(nome_modelo, metricas, y_true, y_pred, y_pred_proba,
                             history=None, diretorio="resultados"):
    """Salva métricas, predições e relatório em arquivos locais."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pasta = os.path.join(diretorio, f"{nome_modelo}_{timestamp}")
    os.makedirs(pasta, exist_ok=True)

    metricas_serial = {
        k: (float(v) if hasattr(v, "item") else v)
        for k, v in metricas.items()
    }

    with open(os.path.join(pasta, "metricas.json"), "w") as f:
        json.dump({"modelo": nome_modelo, "data_execucao": timestamp,
                   "metricas": metricas_serial}, f, indent=4)

    np.save(os.path.join(pasta, "y_true.npy"), y_true)
    np.save(os.path.join(pasta, "y_pred.npy"), y_pred)
    np.save(os.path.join(pasta, "y_pred_proba.npy"), y_pred_proba)

    with open(os.path.join(pasta, "classification_report.json"), "w") as f:
        json.dump(classification_report(y_true, y_pred, output_dict=True), f, indent=4)

    if history is not None:
        hist_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
        with open(os.path.join(pasta, "training_history.json"), "w") as f:
            json.dump(hist_dict, f, indent=4)
        with open(os.path.join(pasta, "training_history.pkl"), "wb") as f:
            pickle.dump(history.history, f)

    with open(os.path.join(pasta, "resumo.txt"), "w", encoding="utf-8") as f:
        f.write("=" * 50 + "\n")
        f.write(f"RESUMO DO MODELO: {nome_modelo}\n")
        f.write("=" * 50 + "\n\n")
        f.write("MÉTRICAS:\n" + "-" * 30 + "\n")
        for metrica, valor in metricas_serial.items():
            f.write(
                f"{metrica}: {valor:.4f}\n" if valor is not None
                else f"{metrica}: Não disponível\n"
            )
        f.write("\nCONFIGURAÇÃO DO MODELO:\n" + "-" * 30 + "\n")
        f.write("Arquitetura: ResNet50\n")
        f.write("Pesos: ImageNet (pré-treinado)\n")
        f.write("Fine-tuning: Camadas base congeladas\n")
        f.write("Carregamento: tf.data lazy (batch a batch)\n")
        f.write("Otimizador: Adam (lr=1e-4)\n")
        f.write("Loss: Sparse Categorical Crossentropy\n")
        f.write("Batch Size: 32\n")
        f.write("Epochs: até 50 (EarlyStopping patience=7)\n")

    print(f"\n✓ Resultados salvos em: {pasta}")
    for arq in ["metricas.json", "y_true/pred.npy", "y_pred_proba.npy",
                "classification_report.json", "training_history.json", "resumo.txt"]:
        print(f"  - {arq}")

    return pasta


# ============================================================
# 4. CONJUNTOS TREINO / VALIDAÇÃO / TESTE
#    Split por grupos e augmentation já feitos na célula base.
#    Validação contém apenas imagens originais (sem vazamento).
# ============================================================

print(f"Treino    : {len(X_tr_paths):>6} imagens")
print(f"Validação : {len(X_val_paths):>6} imagens (somente originais)")
print(f"Teste     : {len(X_test_paths):>6} imagens")

# ============================================================
# 5. DATASETS LAZY
# ============================================================

BATCH_SIZE = 32

train_ds = criar_dataset_resnet(X_tr_paths,    y_tr,    batch_size=BATCH_SIZE, shuffle=True)
val_ds   = criar_dataset_resnet(X_val_paths,   y_val,   batch_size=BATCH_SIZE)
test_ds  = criar_dataset_resnet(X_test_paths,  y_test,  batch_size=BATCH_SIZE)

ram_por_batch = BATCH_SIZE * 224 * 224 * 3 * 4 / 1e6
print(f"\n✓ Datasets criados com carregamento lazy")
print(f"  RAM por batch : ~{ram_por_batch:.0f} MB  (antes: dataset inteiro)")

# ============================================================
# 6. CONSTRUÇÃO DO MODELO
# ============================================================

print("\n" + "=" * 60)
print("CONSTRUÇÃO DO MODELO — ResNet50")
print("=" * 60)

base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

x          = GlobalAveragePooling2D()(base_model.output)
x          = Dropout(0.3)(x)
output     = Dense(len(label_encoder.classes_), activation="softmax")(x)
model_resnet = Model(inputs=base_model.input, outputs=output)

model_resnet.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)

total     = len(model_resnet.layers)
treinavel = len([l for l in model_resnet.layers if l.trainable])
print(f"✓ Total de camadas     : {total}")
print(f"✓ Camadas treináveis   : {treinavel}")
print(f"✓ Camadas congeladas   : {total - treinavel}")

# ============================================================
# 7. TREINAMENTO
# ============================================================

os.makedirs("models", exist_ok=True)

callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=7,
        restore_best_weights=True, verbose=1,
    ),
    ModelCheckpoint(
        "models/resnet50_best.keras", monitor="val_accuracy",
        save_best_only=True, verbose=0,
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3,
        min_lr=1e-7, verbose=1,
    ),
]

print("\n" + "=" * 60)
print("TREINAMENTO — ResNet50")
print("=" * 60)

history = model_resnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
    verbose=1,
)

model_resnet.save("models/resnet50.keras")
print("✓ Modelo salvo em: models/resnet50.keras")

# ============================================================
# 8. AVALIAÇÃO
# ============================================================

print("\n" + "=" * 60)
print("AVALIAÇÃO DO MODELO")
print("=" * 60)

y_pred_proba_resnet = model_resnet.predict(test_ds, verbose=0)
y_pred_resnet       = np.argmax(y_pred_proba_resnet, axis=1)

metricas_resnet = calcular_metricas(
    y_test, y_pred_resnet, y_pred_proba_resnet,
    num_classes=len(label_encoder.classes_),
)

print("\n" + "=" * 50)
print("MÉTRICAS — ResNet50")
print("=" * 50)
for metrica, valor in metricas_resnet.items():
    if valor is not None:
        print(f"  {metrica:<12}: {valor:.4f}")
    else:
        print(f"  {metrica:<12}: Não disponível")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_resnet, target_names=label_encoder.classes_))

# ============================================================
# 9. SALVAR RESULTADOS
# ============================================================

pasta_resultados_resnet = salvar_resultados_resnet(
    nome_modelo="ResNet50",
    metricas=metricas_resnet,
    y_true=y_test,
    y_pred=y_pred_resnet,
    y_pred_proba=y_pred_proba_resnet,
    history=history,
    diretorio="resultados",
)

# ============================================================
# 10. RESUMO FINAL
# ============================================================

print(f"\n{'=' * 60}")
print("✅ RESUMO FINAL — ResNet50")
print(f"{'=' * 60}")
print(f"✓ Acurácia : {metricas_resnet['Accuracy']:.4f}")
print(f"✓ F1-Score : {metricas_resnet['F1-Score']:.4f}")
print(f"✓ AUC      : {metricas_resnet['AUC']:.4f}" if metricas_resnet["AUC"] else "✓ AUC: Não disponível")
print(f"✓ Kappa    : {metricas_resnet['Kappa']:.4f}")
print(f"✓ Pasta    : {pasta_resultados_resnet}")
print(f"\nPróximo passo: Execute a célula de visualização.")

K.clear_session()
gc.collect()
print("\n✓ Sessão Keras limpa")

In [ ]:
# ============================================
# VISUALIZAÇÃO DOS RESULTADOS - ResNet50
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import json
import os
from pathlib import Path

# Configurar estilo dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# ============================================
# 1. CARREGAR RESULTADOS SALVOS
# ============================================

# ============================================
# ENCONTRAR AUTOMATICAMENTE A PASTA DA RESNET50
# ============================================

import os
import glob

def encontrar_pasta_resultados(nome_modelo="ResNet50", diretorio_base="resultados"):
    """
    Encontra automaticamente a pasta de resultados mais recente para um modelo.
    """
    # Listar todas as pastas que começam com o nome do modelo
    padrao = os.path.join(diretorio_base, f"{nome_modelo}_*")
    pastas = sorted(glob.glob(padrao))

    if not pastas:
        raise FileNotFoundError(
            f"Nenhuma pasta encontrada para {nome_modelo} em {diretorio_base}/\n"
            f"Verifique se o treinamento foi executado e os resultados foram salvos."
        )

    # Retornar a pasta mais recente (última em ordem alfabética = mais recente)
    pasta_mais_recente = pastas[-1]

    print(f"📁 Pastas encontradas para {nome_modelo}:")
    for i, pasta in enumerate(pastas, 1):
        if pasta == pasta_mais_recente:
            print(f"  {i}. {os.path.basename(pasta)} ← SELECIONADA (mais recente)")
        else:
            print(f"  {i}. {os.path.basename(pasta)}")

    return pasta_mais_recente

# Encontrar pasta automaticamente
pasta_resultados = encontrar_pasta_resultados("ResNet50")
print(f"\n✓ Usando: {pasta_resultados}")

def carregar_resultados(pasta_resultados):
    """Carrega os resultados salvos do modelo"""
    resultados = {}

    # Carregar métricas
    with open(os.path.join(pasta_resultados, "metricas.json"), 'r') as f:
        resultados['metricas'] = json.load(f)

    # Carregar arrays numpy
    resultados['y_true'] = np.load(os.path.join(pasta_resultados, "y_true.npy"))
    resultados['y_pred'] = np.load(os.path.join(pasta_resultados, "y_pred.npy"))
    resultados['y_pred_proba'] = np.load(os.path.join(pasta_resultados, "y_pred_proba.npy"))

    # Carregar classification report
    with open(os.path.join(pasta_resultados, "classification_report.json"), 'r') as f:
        resultados['classification_report'] = json.load(f)

    # Carregar histórico se existir
    history_path = os.path.join(pasta_resultados, "training_history.json")
    if os.path.exists(history_path):
        with open(history_path, 'r') as f:
            resultados['history'] = json.load(f)

    return resultados

# Carregar dados
print("Carregando resultados salvos...")
resultados = carregar_resultados(pasta_resultados)
print("✓ Resultados carregados com sucesso!")

# ============================================
# 2. TABELA DE MÉTRICAS
# ============================================
print("\n" + "="*60)
print("📊 TABELA DE MÉTRICAS - ResNet50")
print("="*60)

# Criar DataFrame com as métricas
metricas_dict = resultados['metricas']['metricas']
df_metricas = pd.DataFrame({
    'Métrica': list(metricas_dict.keys()),
    'Valor': list(metricas_dict.values())
})

# Formatar valores
def format_bar(value):
    """Cria uma barra de progresso visual"""
    if value is None or value == "N/A":
        return "Não disponível"
    try:
        val = float(value)
        bar_length = int(val * 20)  # 20 caracteres máximos
        bar = "█" * bar_length + "░" * (20 - bar_length)
        return f"{bar} {val:.4f}"
    except:
        return str(value)

df_metricas['Visualização'] = df_metricas['Valor'].apply(format_bar)

# Exibir tabela formatada
print("\nMÉTRICAS DE DESEMPENHO:")
print("-"*60)
for idx, row in df_metricas.iterrows():
    print(f"  {row['Métrica']:<12} {row['Visualização']}")
print("-"*60)

# ============================================
# 3. MATRIZ DE CONFUSÃO
# ============================================
print("\n" + "="*60)
print("🎯 MATRIZ DE CONFUSÃO - ResNet50")
print("="*60)

# Obter classes
if hasattr(label_encoder, 'classes_'):
    nomes_classes = [str(c) for c in label_encoder.classes_]
else:
    # Fallback: deduzir classes únicas
    nomes_classes = sorted(set(str(c) for c in np.unique(np.concatenate([resultados['y_true'], resultados['y_pred']]))))

n_classes = len(nomes_classes)
print(f"Classes detectadas: {n_classes}")
print(f"Nomes: {', '.join(nomes_classes[:5])}{'...' if len(nomes_classes) > 5 else ''}")

# Calcular matriz de confusão
cm = confusion_matrix(resultados['y_true'], resultados['y_pred'])

# Criar figura com subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Para muitas classes, ajustar tamanho da fonte
if n_classes > 10:
    annot_fontsize = 8
    rotation_angle = 45
else:
    annot_fontsize = 10
    rotation_angle = 0

# ===== Matriz de Confusão - Contagens =====
sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=nomes_classes,
            yticklabels=nomes_classes,
            ax=axes[0],
            annot_kws={'size': annot_fontsize},
            cbar_kws={'label': 'Contagem'})
axes[0].set_title('Matriz de Confusão - Contagens', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Classe Predita', fontsize=12)
axes[0].set_ylabel('Classe Verdadeira', fontsize=12)
axes[0].tick_params(axis='x', rotation=rotation_angle)
axes[0].tick_params(axis='y', rotation=0)

# ===== Matriz de Confusão - Percentuais =====
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_percent,
            annot=True,
            fmt='.1f',
            cmap='RdYlGn',
            xticklabels=nomes_classes,
            yticklabels=nomes_classes,
            ax=axes[1],
            vmin=0,
            vmax=100,
            annot_kws={'size': annot_fontsize},
            cbar_kws={'label': '%'})
axes[1].set_title('Matriz de Confusão - Percentuais (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Classe Predita', fontsize=12)
axes[1].set_ylabel('Classe Verdadeira', fontsize=12)
axes[1].tick_params(axis='x', rotation=rotation_angle)
axes[1].tick_params(axis='y', rotation=0)

plt.suptitle('ResNet50 - Análise de Classificação', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(pasta_resultados, 'matriz_confusao.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Matriz de confusão salva em: matriz_confusao.png")

# ============================================
# 4. MÉTRICAS POR CLASSE (CORRIGIDO)
# ============================================
print("\n" + "="*60)
print("📈 MÉTRICAS POR CLASSE")
print("="*60)

# Extrair métricas por classe do classification report
report = resultados['classification_report']

# Filtrar apenas as entradas numéricas (classes)
classes_no_report = [k for k in report.keys() if k not in ['accuracy', 'macro avg', 'weighted avg'] and isinstance(report[k], dict)]

if not classes_no_report:
    print("⚠️ Nenhuma classe individual encontrada no report. Usando índices numéricos...")
    # Criar mapeamento se necessário
    classes_no_report = [str(c) for c in sorted(set(resultados['y_true']))]

    # Recalcular métricas por classe se não estiverem no report
    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, support = precision_recall_fscore_support(
        resultados['y_true'],
        resultados['y_pred'],
        average=None
    )

    report_por_classe = {}
    for i, classe in enumerate(classes_no_report):
        report_por_classe[classe] = {
            'precision': precision[i] if i < len(precision) else 0,
            'recall': recall[i] if i < len(recall) else 0,
            'f1-score': f1[i] if i < len(f1) else 0,
            'support': support[i] if i < len(support) else 0
        }
else:
    report_por_classe = {classe: report[classe] for classe in classes_no_report}

# Criar DataFrame com métricas por classe
df_por_classe = pd.DataFrame()
for classe, metrics in report_por_classe.items():
    df_por_classe[classe] = {
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1-Score': metrics['f1-score'],
        'Support': metrics['support']
    }

df_por_classe = df_por_classe.round(3)

# Adicionar médias
for avg_type in ['macro avg', 'weighted avg']:
    if avg_type in report:
        df_por_classe[avg_type] = {
            'Precision': report[avg_type]['precision'],
            'Recall': report[avg_type]['recall'],
            'F1-Score': report[avg_type]['f1-score'],
            'Support': report[avg_type]['support']
        }

print("\nTabela de métricas por classe:")
print("="*60)
# Mostrar apenas as primeiras e últimas classes se houver muitas
if len(df_por_classe.columns) > 10:
    print("(Mostrando primeiras 5 e últimas 5 classes)")
    pd.set_option('display.max_columns', 12)
print(df_por_classe.to_string())
print("="*60)

# ============================================
# 5. RESUMO VISUAL (CORRIGIDO)
# ============================================
print("\n" + "="*60)
print("🎯 RESUMO DO DESEMPENHO")
print("="*60)

# Encontrar melhor e pior classe usando o report_por_classe
if report_por_classe:
    # Ordenar classes por F1-Score
    f1_scores = {classe: metrics['f1-score'] for classe, metrics in report_por_classe.items()}
    if f1_scores:
        melhor_classe = max(f1_scores, key=f1_scores.get)
        pior_classe = min(f1_scores, key=f1_scores.get)

        print(f"✓ Acurácia geral: {metricas_dict['Accuracy']:.4f}")
        print(f"✓ Número de classes: {len(report_por_classe)}")
        print(f"✓ Melhor classe: {melhor_classe} (F1-Score: {f1_scores[melhor_classe]:.3f})")
        print(f"✓ Pior classe: {pior_classe} (F1-Score: {f1_scores[pior_classe]:.3f})")
        print(f"✓ Total de imagens de teste: {len(resultados['y_true'])}")

        # Distribuição das classes
        print(f"\nDistribuição - Top 5 classes com melhor F1-Score:")
        top5 = sorted(f1_scores.items(), key=lambda x: x[1], reverse=True)[:5]
        for i, (classe, f1) in enumerate(top5, 1):
            print(f"  {i}. {classe}: {f1:.3f}")

        print(f"\nDistribuição - Top 5 classes com pior F1-Score:")
        bottom5 = sorted(f1_scores.items(), key=lambda x: x[1])[:5]
        for i, (classe, f1) in enumerate(bottom5, 1):
            print(f"  {i}. {classe}: {f1:.3f}")
else:
    print("⚠️ Não foi possível extrair métricas por classe.")

# Salvar tabelas como CSV
df_metricas.to_csv(os.path.join(pasta_resultados, 'tabela_metricas.csv'), index=False)
df_por_classe.to_csv(os.path.join(pasta_resultados, 'metricas_por_classe.csv'))

print(f"\n✓ Tabelas salvas em: {pasta_resultados}")
print(f"  - tabela_metricas.csv")
print(f"  - metricas_por_classe.csv")
print(f"  - matriz_confusao.png")

# ============================================
# 6. GRÁFICO DE BARRAS DAS MÉTRICAS
# ============================================
print("\n" + "="*60)
print("📊 GERANDO GRÁFICO DE BARRAS")
print("="*60)

# Filtrar apenas métricas numéricas
metricas_plot = {k: v for k, v in metricas_dict.items() if v is not None}

if metricas_plot:
    fig, ax = plt.subplots(figsize=(10, 6))

    # Criar barras com gradiente de cores
    n_barras = len(metricas_plot)
    cores = plt.cm.viridis(np.linspace(0.2, 0.9, n_barras))
    barras = ax.bar(range(n_barras),
                    list(metricas_plot.values()),
                    color=cores,
                    edgecolor='black',
                    linewidth=1.5)

    # Adicionar valores nas barras
    for barra, (nome, valor) in zip(barras, metricas_plot.items()):
        height = barra.get_height()
        ax.text(barra.get_x() + barra.get_width()/2., height + 0.01,
                f'{valor:.4f}',
                ha='center', va='bottom', fontweight='bold', fontsize=11)

    # Configurar eixos
    ax.set_xticks(range(n_barras))
    ax.set_xticklabels(list(metricas_plot.keys()), rotation=45, ha='right')
    ax.set_ylabel('Valor', fontsize=12, fontweight='bold')
    ax.set_title('DenseNet121 - Métricas de Desempenho', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)

    # Adicionar linhas de referência
    ax.axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Excelente (0.8)')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Bom (0.6)')
    ax.axhline(y=0.4, color='red', linestyle='--', alpha=0.5, label='Regular (0.4)')
    ax.legend(loc='lower left')

    plt.tight_layout()
    plt.savefig(os.path.join(pasta_resultados, 'grafico_metricas.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Gráfico de métricas salvo em: grafico_metricas.png")
else:
    print("⚠️ Nenhuma métrica disponível para plotar.")

print("\n✅ Visualização completa dos resultados!")

## ConvNeXt V2

CNN moderna. Use este bloco para o terceiro experimento.

In [ ]:
# ============================================================
# ConvNeXt V2 PURA — Classificação Fine-Tuning (CORRIGIDO)
# Correções aplicadas:
#   FIX 1 — Freeze de backbone + differential learning rate
#   FIX 2 — Eliminação de dupla augmentação (offline + online)
#   FIX 3 — Label smoothing no CrossEntropyLoss
#   FIX 4 — Split por grupos na célula base (sem vazamento treino/val)
#   FIX 5 — Dropout explícito no head (drop_rate=0.3)
#   FIX 6 — Augmentação online diversificada para originais
#   FIX 7 — Early stopping com paciência
#   FIX 8 — weight_decay explícito no AdamW
#   FIX 9 — Gradient clipping (max_norm=1.0)
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================
import os
import gc
import json
import numpy as np
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from PIL import Image
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    roc_auc_score,
    classification_report,
)
from sklearn.preprocessing import label_binarize

# ============================================================
# 2. CONFIGURAÇÕES GERAIS
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando device: {device}")

BATCH_SIZE     = 32
EPOCHS         = 10     # FIX 7: ampliado; early stopping interrompe no momento certo
LR             = 1e-4
NUM_CLASSES    = len(label_encoder.classes_)
IMG_SIZE       = 224
N_FREEZE_EPOCHS = 2     # FIX 1: épocas com backbone congelado (Fase 1)
PATIENCE       = 4      # FIX 7: épocas sem melhora antes de parar

# ============================================================
# 3. CONJUNTOS TREINO / VALIDAÇÃO / TESTE
#    Split por grupos e augmentation já feitos na célula base.
#    Validação = somente imagens originais (sem vazamento).
# ============================================================

X_train_split = X_tr_paths
y_train_split = y_tr
X_val_split   = X_val_paths
y_val_split   = y_val

print(f"Treino após split:      {len(X_train_split)} amostras")
print(f"Validação após split:   {len(X_val_split)} amostras (somente originais)")
print(f"Teste (intocado):       {len(X_test_paths)} amostras")

# ============================================================
# 4. DATASET — CORRIGE DUPLA AUGMENTAÇÃO (FIX 2 + FIX 6)
#
# Problema original: o dataset já contém imagens pré-geradas
# offline (_hflip, _rot30, etc.) e o transform_train aplicava
# RandomHorizontalFlip + RandomRotation em cima delas,
# criando imagens quase idênticas e forçando o modelo a
# memorizar variações específicas em vez de generalizar.
#
# Solução: SmartImageDataset detecta se a imagem é original
# ou já aumentada e aplica o transform adequado:
#   - Originais → augmentação diversificada (FIX 6)
#   - Pré-aumentadas → apenas resize + normalize (sem dupla aug)
#   - Val / Teste → apenas resize + normalize
# ============================================================

SUFIXOS_AUG = ["_hflip", "_vflip", "_rot30", "_jitter", "_blur", "_affine"]

def is_pre_augmented(path: str) -> bool:
    """Retorna True se o arquivo já é uma versão aumentada offline."""
    nome = os.path.splitext(os.path.basename(path))[0]
    return any(nome.endswith(s) for s in SUFIXOS_AUG)

# FIX 6: Augmentação online para imagens ORIGINAIS usa apenas
# transformações que NÃO existem no conjunto offline, aumentando
# a diversidade real de treino sem criar redundância.
transform_originais = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.4),
    transforms.RandomGrayscale(p=0.05),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),          # só após ToTensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Imagens já aumentadas offline recebem só normalização:
transform_pre_aug = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Validação e Teste nunca recebem augmentação:
transform_test = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class SmartImageDataset(Dataset):
    """
    Dataset que evita dupla augmentação (FIX 2):
    - Se transform_fixed for fornecido, aplica-o a todas as imagens
      (usar em val/test para garantir consistência).
    - Caso contrário, distingue imagens originais de pré-aumentadas
      e aplica o transform correspondente.
    """

    def __init__(self, paths, labels,
                 transform_orig=None,
                 transform_aug=None,
                 transform_fixed=None):
        self.paths           = paths
        self.labels          = labels
        self.transform_orig  = transform_orig   # para imagens originais
        self.transform_aug   = transform_aug    # para imagens pré-aumentadas
        self.transform_fixed = transform_fixed  # override único (val/test)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform_fixed is not None:
            img = self.transform_fixed(img)
        elif is_pre_augmented(self.paths[idx]):
            if self.transform_aug:
                img = self.transform_aug(img)
        else:
            if self.transform_orig:
                img = self.transform_orig(img)
        return img, self.labels[idx]


train_dataset = SmartImageDataset(
    X_train_split, y_train_split,
    transform_orig=transform_originais,
    transform_aug=transform_pre_aug,
)
val_dataset = SmartImageDataset(
    X_val_split, y_val_split,
    transform_fixed=transform_test,
)
test_dataset = SmartImageDataset(
    X_test_paths, y_test,
    transform_fixed=transform_test,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

# ============================================================
# 5. MODELO ConvNeXt V2
#    FIX 1: Freeze de backbone + differential LR em duas fases
#    FIX 5: drop_rate=0.3 — dropout explícito no head
# ============================================================

# FIX 5: drop_rate adiciona Dropout antes da camada linear final
model = timm.create_model(
    "convnextv2_tiny",
    pretrained=True,
    num_classes=NUM_CLASSES,
    drop_rate=0.3,          # FIX 5: era 0.0 (padrão oculto)
)
model = model.to(device)


# FIX 1 — Helpers de fase

def freeze_backbone(model: nn.Module) -> None:
    """Fase 1: congela tudo exceto o head classificador."""
    for name, param in model.named_parameters():
        param.requires_grad = "head" in name
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Fase 1 — parâmetros treináveis: {trainable:,} (apenas head)")


def unfreeze_all(model: nn.Module) -> None:
    """Fase 2: descongela todo o modelo."""
    for param in model.parameters():
        param.requires_grad = True
    total = sum(p.numel() for p in model.parameters())
    print(f"  Fase 2 — parâmetros treináveis: {total:,} (modelo completo)")


def make_optimizer(model: nn.Module, phase: int) -> torch.optim.Optimizer:
    """
    FIX 1 + FIX 8: Otimizador com weight_decay=1e-4 explícito.
    Fase 1: apenas head com LR=1e-4.
    Fase 2: backbone com LR/10, head com LR (differential LR).
    """
    if phase == 1:
        params = filter(lambda p: p.requires_grad, model.parameters())
        return torch.optim.AdamW(params, lr=LR, weight_decay=1e-4)
    else:
        backbone_params = [p for n, p in model.named_parameters()
                           if "head" not in n and p.requires_grad]
        head_params     = [p for n, p in model.named_parameters()
                           if "head" in n     and p.requires_grad]
        return torch.optim.AdamW([
            {"params": backbone_params, "lr": LR / 10},  # backbone: 1e-5
            {"params": head_params,     "lr": LR},        # head:     1e-4
        ], weight_decay=1e-4)                             # FIX 8: explícito


# FIX 3: label_smoothing=0.1 evita logits → +∞ e suaviza a confiança
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Inicializa Fase 1
freeze_backbone(model)
optimizer = make_optimizer(model, phase=1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_FREEZE_EPOCHS
)

# ============================================================
# 6. FUNÇÕES DE TREINO E AVALIAÇÃO
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        # FIX 9: gradient clipping — limita a norma dos gradientes
        #        antes do passo do otimizador
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc  = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * inputs.size(0)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss, np.array(all_preds), np.array(all_labels), np.array(all_probs)


def calcular_metricas(y_true, y_pred, y_proba):
    metricas = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall":    recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1-Score":  f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "Kappa":     cohen_kappa_score(y_true, y_pred),
    }
    y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
    metricas["AUC"] = roc_auc_score(
        y_true_bin, y_proba, multi_class="ovr", average="weighted"
    )
    return metricas

# ============================================================
# 7. LOOP DE TREINAMENTO
#    FIX 1: transição automática de Fase 1 → Fase 2
#    FIX 7: early stopping com paciência
# ============================================================
print("\n" + "=" * 60)
print("TREINAMENTO ConvNeXt V2 (corrigido)")
print("=" * 60)
print(f"Fase 1: épocas 1–{N_FREEZE_EPOCHS} (backbone congelado)")
print(f"Fase 2: épocas {N_FREEZE_EPOCHS+1}–{EPOCHS} (differential LR)")
print(f"Early stopping: paciência = {PATIENCE} épocas\n")

os.makedirs("models", exist_ok=True)

best_acc   = 0.0
wait       = 0          # contador de épocas sem melhora (FIX 7)
fase_atual = 1

for epoch in range(EPOCHS):

    # ── FIX 1: Transição Fase 1 → Fase 2 ──────────────────────
    if epoch == N_FREEZE_EPOCHS and fase_atual == 1:
        print(f"\n{'─'*60}")
        print(f"Época {epoch+1}: iniciando Fase 2 — descongelando backbone")
        unfreeze_all(model)
        optimizer = make_optimizer(model, phase=2)
        # Scheduler recomeça para as épocas restantes da Fase 2
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - N_FREEZE_EPOCHS
        )
        fase_atual = 2
        print(f"{'─'*60}\n")
    # ───────────────────────────────────────────────────────────

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, y_pred_val, y_true_val, y_proba_val = evaluate(
        model, val_loader, criterion, device
    )
    val_metrics = calcular_metricas(y_true_val, y_pred_val, y_proba_val)
    scheduler.step()

    print(
        f"Época {epoch+1:02d}/{EPOCHS} [F{fase_atual}] | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_metrics['Accuracy']:.4f} | "
        f"Val F1: {val_metrics['F1-Score']:.4f} | Val AUC: {val_metrics['AUC']:.4f}"
    )

    # ── Checkpoint do melhor modelo ────────────────────────────
    if val_metrics["Accuracy"] > best_acc:
        best_acc = val_metrics["Accuracy"]
        wait = 0
        torch.save(model.state_dict(), "models/convnextv2_puro_best.pth")
        print(f"  ✓ Melhor modelo salvo (Val Acc: {best_acc:.4f})")
    else:
        wait += 1
        print(f"  – Sem melhora ({wait}/{PATIENCE})")

        # ── FIX 7: Early stopping ──────────────────────────────
        if wait >= PATIENCE:
            print(f"\n  Early stopping na época {epoch+1} "
                  f"(paciência={PATIENCE} esgotada). "
                  f"Melhor Val Acc: {best_acc:.4f}")
            break
        # ───────────────────────────────────────────────────────

# ============================================================
# 8. AVALIAÇÃO FINAL (CONJUNTO DE TESTE)
# ============================================================
print("\n" + "=" * 60)
print("AVALIAÇÃO FINAL (conjunto de teste)")
print("=" * 60)

model.load_state_dict(
    torch.load("models/convnextv2_puro_best.pth",
               map_location=device, weights_only=True)
)
test_loss, y_pred_test, y_true_test, y_proba_test = evaluate(
    model, test_loader, criterion, device
)
metricas_finais = calcular_metricas(y_true_test, y_pred_test, y_proba_test)

print("\nMÉTRICAS FINAIS (Teste):")
for k, v in metricas_finais.items():
    print(f"  {k:<12}: {v:.4f}")
print("\nClassification Report (Teste):")
print(classification_report(
    y_true_test, y_pred_test, target_names=label_encoder.classes_
))

# ============================================================
# 9. SALVAR RESULTADOS
# ============================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
pasta = os.path.join("resultados", f"ConvNeXtV2_Puro_Corrigido_{timestamp}")
os.makedirs(pasta, exist_ok=True)

with open(os.path.join(pasta, "metricas.json"), "w") as f:
    json.dump(
        {"modelo": "ConvNeXtV2_Puro_Corrigido", "metricas": metricas_finais},
        f, indent=4
    )

np.save(os.path.join(pasta, "y_true.npy"),  y_true_test)
np.save(os.path.join(pasta, "y_pred.npy"),  y_pred_test)
np.save(os.path.join(pasta, "y_proba.npy"), y_proba_test)

with open(os.path.join(pasta, "classification_report.json"), "w") as f:
    json.dump(
        classification_report(y_true_test, y_pred_test, output_dict=True),
        f, indent=4
    )

print(f"\n✓ Resultados salvos em: {pasta}")
gc.collect()
print("✓ Memória liberada")

In [ ]:
# ============================================================
# VISUALIZAÇÕES DOS RESULTADOS — ConvNeXt V2 Puro
# Variáveis esperadas (definidas na célula de treinamento):
#   y_test         → labels reais do conjunto de teste
#   y_pred_test    → predições do modelo
#   y_proba_test   → probabilidades softmax (shape: n_amostras × n_classes)
#   metricas_finais → dict com Accuracy, Precision, Recall, F1-Score, Kappa, AUC
#   pasta          → caminho onde os resultados já foram salvos
#   label_encoder  → LabelEncoder ajustado na base
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from itertools import cycle

# Configuração de estilo
sns.set_style("whitegrid")
plt.rcParams['figure.dpi']    = 100
plt.rcParams['savefig.dpi']   = 300
plt.rcParams['font.size']     = 12

class_names = label_encoder.classes_
n_classes   = len(class_names)

print("\n" + "=" * 60)
print("GERANDO VISUALIZAÇÕES — ConvNeXt V2 Puro")
print("=" * 60)

# ------------------------------------------------------------
# 1. MATRIZ DE CONFUSÃO
# ------------------------------------------------------------
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Contagem'})
plt.title('Matriz de Confusão — ConvNeXt V2 Puro', fontsize=14, pad=15)
plt.xlabel('Predito', fontsize=12)
plt.ylabel('Real', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(pasta, 'matriz_confusao.png'), bbox_inches='tight')
plt.show()
print("✓ Matriz de confusão salva e exibida")

# ------------------------------------------------------------
# 2. TABELA DE MÉTRICAS
# ------------------------------------------------------------
df_metrics = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Kappa', 'AUC'],
    'Valor': [
        metricas_finais['Accuracy'],
        metricas_finais['Precision'],
        metricas_finais['Recall'],
        metricas_finais['F1-Score'],
        metricas_finais['Kappa'],
        metricas_finais['AUC'],
    ]
})
df_metrics['Valor'] = df_metrics['Valor'].apply(lambda x: f"{x:.4f}")

print("\nTabela de Métricas:")
print("=" * 30)
print(df_metrics.to_string(index=False))
print("=" * 30)

df_metrics.to_csv(os.path.join(pasta, 'tabela_metricas.csv'), index=False)
print("✓ Tabela de métricas salva como CSV")

# ------------------------------------------------------------
# 3. GRÁFICO DE BARRAS DAS MÉTRICAS
# ------------------------------------------------------------
metric_names  = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Kappa', 'AUC']
metric_values = [metricas_finais[m] for m in metric_names]
colors        = sns.color_palette("viridis", len(metric_names))

plt.figure(figsize=(10, 6))
bars = plt.bar(metric_names, metric_values,
               color=colors, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, metric_values):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

plt.ylim(0, 1.1)
plt.title('Métricas de Desempenho — ConvNeXt V2 Puro', fontsize=14, pad=15)
plt.ylabel('Valor')
plt.xlabel('Métrica')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(pasta, 'barras_metricas.png'), bbox_inches='tight')
plt.show()
print("✓ Gráfico de barras salvo e exibido")

# ------------------------------------------------------------
# 4. CURVA ROC (MULTICLASSE — ONE-VS-REST)
#    O ConvNeXt puro sempre retorna probabilidades softmax,
#    então não há verificação de decision_function necessária.
# ------------------------------------------------------------
y_test_bin  = label_binarize(y_test, classes=range(n_classes))
fpr         = {}
tpr         = {}
roc_auc_per = {}

plt.figure(figsize=(10, 8))
colors_roc = cycle(sns.color_palette("bright", n_classes))

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba_test[:, i])
    roc_auc_per[i]    = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], color=next(colors_roc), lw=2,
             label=f'{class_names[i]} (AUC = {roc_auc_per[i]:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.7, label='Aleatório (AUC = 0.500)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taxa de Falsos Positivos (1 − Especificidade)', fontsize=12)
plt.ylabel('Taxa de Verdadeiros Positivos (Sensibilidade)', fontsize=12)
plt.title('Curvas ROC (One-vs-Rest) — ConvNeXt V2 Puro', fontsize=14, pad=15)
plt.legend(loc="lower right", fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(pasta, 'curva_roc.png'), bbox_inches='tight')
plt.show()
print("✓ Curvas ROC salvas e exibidas")

print("\n" + "=" * 60)
print("✅ TODAS AS VISUALIZAÇÕES FORAM GERADAS COM SUCESSO!")
print(f"   Arquivos salvos em: {pasta}")
print("=" * 60)

## Carregar e testar modelos salvos

Use estas células para carregar os modelos treinados e fazer previsões sem retreinar.

In [ ]:
# Comparação resumida

# Carregar os arrays salvos se as variáveis não estiverem na memória
import numpy as np
import glob

def carregar_ultima_predicao(nome_modelo):
    """Carrega a última predição salva para um modelo"""
    pasta = f"resultados/{nome_modelo}_*"
    pastas = sorted(glob.glob(pasta))
    if not pastas:
        return None
    ultima = pastas[-1]
    y_pred = np.load(f"{ultima}/y_pred.npy")
    return y_pred

# Tenta carregar do ambiente ou dos arquivos salvos
try:
    y_pred_densenet = y_pred  # do bloco DenseNet121
except NameError:
    y_pred_densenet = carregar_ultima_predicao("DenseNet121")

try:
    y_pred_resnet = y_pred_resnet  # do bloco ResNet50
except NameError:
    y_pred_resnet = carregar_ultima_predicao("ResNet50")

# ConvNeXt V2 já está na memória como y_pred_convnext
try:
    y_pred_convnext = y_pred_convnext
except NameError:
    y_pred_convnext = carregar_ultima_predicao("ConvNextV2_SVM")

# Montar dicionário com apenas os que foram carregados
resultados_carregados = {}
if y_pred_densenet is not None:
    resultados_carregados["DenseNet121"] = accuracy_score(y_test, y_pred_densenet)
if y_pred_resnet is not None:
    resultados_carregados["ResNet50"] = accuracy_score(y_test, y_pred_resnet)
if y_pred_convnext is not None:
    resultados_carregados["ConvNeXt V2"] = accuracy_score(y_test, y_pred_convnext)

print("\n=== RESUMO DE RESULTADOS ===")
for modelo, acc in resultados_carregados.items():
    print(f"{modelo:20s}: {acc:.4f}")